# Exercice 05 – Résumer des PV en parallèle

Le dossier `pv_ca/` contient 20 retranscriptions de conseils d'administration (PDF texte, 5 à 9 pages). On veut les résumer automatiquement avec `llama-server`, en envoyant plusieurs documents **en même temps**, et mesurer ce que ça coûte.

## Ce qui est attendu

### 1. Une fonction `resumer(chemin_pdf)`

Elle doit :

- lire le texte du PDF ;
- l'envoyer au modèle avec une **sortie structurée** (schéma Pydantic) contenant au minimum :
  - un résumé de la séance (150 à 250 mots),
  - la liste des décisions : numéro du point, intitulé, statut (`approuvé`, `rejeté`, `reporté`, `retiré`, `information`), votes pour / contre / abstentions,
  - la date de la prochaine réunion ;
- renvoyer le résultat **et les mesures** : durée de l'appel mesurée côté client, et, dans les `timings` du serveur, le nombre de tokens du prompt, le temps de traitement du prompt, le nombre de tokens générés et la vitesse de génération.

### 2. Traiter les 20 PV en parallèle

Appeler `resumer` sur les 20 fichiers avec un `ThreadPoolExecutor`, avec **4 clients simultanés**.

Afficher :

- un tableau avec une ligne par PV : durée, tokens du prompt, tokens générés, tokens/s ;
- le total : durée de l'ensemble et nombre de PV traités par minute.

### 3. (Optionnel) Comparer des configurations du serveur

Relancer `llama-server` avec d'autres réglages, refaire l'étape 2, et comparer la durée totale, les PV/minute et la VRAM (`nvidia-smi`). Pistes :

- nombre de slots `-np` : 1, 2, 4, 8 ;
- nombre de clients (`max_workers`) : 1, 4, 8 ;
- type du KV cache `-ctk` / `-ctv` : `f16`, `q8_0`, `q4_0` ;
- taille du micro-batch `-ub` ;
- `--spec-type ngram-mod`.

Conclure en quelques lignes : quelle configuration est la plus rapide, et pourquoi ?

## À savoir

- `-c` est le contexte **total**, partagé entre les slots : avec `-np 4 -c 49152`, chaque slot dispose de 12 288 tokens. Un PV fait jusqu'à ~7 000 tokens, et il faut ajouter la réponse.
- Si `max_workers` dépasse `-np`, les requêtes en trop attendent leur tour sur le serveur.
- `--reasoning off` au lancement du serveur évite le mode thinking.
- Le fichier `verite_pv.json` contient les décisions attendues pour chaque PV, si vous voulez vérifier la qualité des réponses.

In [ ]:
ici